# FACULTY SOLUTION — Week 12: What the company should become — transition portfolio
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['envelope', 'buckets', 'projects', 'carbon_scenarios', 'demand_scenarios', 'worked_example_projects']}

In [ ]:

def compute(d):
    import itertools
    pj = d['projects'].set_index('project'); bk = d['buckets'].set_index('bucket'); env = {x.parameter: x.value for x in d['envelope'].itertuples()}
    r = {'discretionary': env['total_envelope'] - env['sustaining_floor']}
    for p in pj.index:
        vals = [pj.loc[p, 'npv_base'] + pj.loc[p, 'carbon_sens'] * (c - 40) + pj.loc[p, 'demand_sens'] * k
                for c in d['carbon_scenarios'].carbon_price for k in d['demand_scenarios'].code]
        r['npv_min_' + p] = min(vals); r['npv_max_' + p] = max(vals)
        r['npv_lowcarbon_slow_' + p] = pj.loc[p, 'npv_base']
        r['npv_highcarbon_collapse_' + p] = pj.loc[p, 'npv_base'] + pj.loc[p, 'carbon_sens'] * 140 + pj.loc[p, 'demand_sens'] * 2
    names = list(pj.index)
    def feasible(sel, adj_ceiling, proceeds):
        envl = r['discretionary'] + (proceeds if 'euro_retail_divest' in sel else 0)
        spend = sum(pj.loc[p, 'cost_musd'] for p in sel if pj.loc[p, 'cost_musd'] > 0)
        by = {}
        for p in sel:
            if pj.loc[p, 'cost_musd'] > 0: by[pj.loc[p, 'bucket']] = by.get(pj.loc[p, 'bucket'], 0) + pj.loc[p, 'cost_musd']
        ceil = dict(bk['ceiling']); ceil['adjacent'] = adj_ceiling
        return spend <= envl and all(v <= ceil[b] for b, v in by.items())
    def analyse(adj_ceiling, proceeds):
        sets = [set(s) for n in range(1, len(names) + 1) for s in itertools.combinations(names, n)]
        feas = [s for s in sets if feasible(s, adj_ceiling, proceeds)]
        hr = [s for s in feas if 'helix_rotterdam' in s]
        unlock = [s for s in feas if 'euro_retail_divest' in s and not feasible(s - {'euro_retail_divest'}, adj_ceiling, proceeds) and len(s) > 1]
        return len(feas), len(hr), len(unlock)
    cur = analyse(bk.loc['adjacent', 'ceiling'], -pj.loc['euro_retail_divest', 'cost_musd'])
    r['feasible_portfolios'], r['feasible_with_helix_rotterdam'], r['portfolios_unlocked_by_divest'] = cur
    envd = r['discretionary'] - pj.loc['euro_retail_divest', 'cost_musd']
    r['hr_plus_wind_cost'] = pj.loc['helix_rotterdam', 'cost_musd'] + pj.loc['offshore_wind', 'cost_musd']
    r['envelope_with_divest'] = envd
    r['hr_plus_wind_needs_divest'] = (r['hr_plus_wind_cost'] > r['discretionary']) and (r['hr_plus_wind_cost'] <= envd)
    r['helix_rotterdam_cost'] = pj.loc['helix_rotterdam', 'cost_musd']; r['adjacent_ceiling'] = bk.loc['adjacent', 'ceiling']
    return r


In [ ]:
fx = json.load(open('../fixtures/week12_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')